# Reviews Analyzer — лабораторная №1: правила и реальные LLM-пробы
Евгений Агарков · AI Engineer · 21.11.2026.

Выполнены локальные правила и Qwen3-4B-4bit: 180 основных генераций и один smoke.
Этот ноутбук повторяет правила и анализирует сохранённые реальные ответы LLM.
Повторная генерация запускается отдельной командой; сетевых вызовов в ячейках нет.
Наборы использованы при разработке; независимой проверки нет.


In [1]:
from pathlib import Path
import json, sys, tempfile
root = Path.cwd().resolve()
while not (root / "experiments/lab1/run.py").exists():
    if root == root.parent:
        raise RuntimeError("Откройте ноутбук внутри репозитория")
    root = root.parent
sys.path.insert(0, str(root / "experiments/lab1"))
from run import read_jsonl, run_experiment, DATA
records = read_jsonl(DATA)
print("Golden:", len(records), "отзывов;", sum(len(r["expected"]["problems"]) for r in records), "проблем")
print("product_id есть:", all("product_id" in r["input"] for r in records))
print("Группировка внутри товара: нет достаточных данных")


Golden: 12 отзывов; 12 проблем
product_id есть: False
Группировка внутри товара: нет достаточных данных


## Воспроизведение
Результаты сохраняются во временный каталог: исходные артефакты сдачи остаются доступны.
Время CPU-правил не является временем LLM или полного приложения.


In [2]:
temporary = Path(tempfile.mkdtemp(prefix="sii-lab1-notebook-"))
reports = {}
for version in ["rules-v0", "rules-v1"]:
    reports[version] = run_experiment("rules", version, out=temporary)
reports["challenge"] = run_experiment("rules", "rules-v1",
    dataset=root / "experiments/lab1/challenge-v0.1.jsonl", out=temporary / "challenge")
for name, report in reports.items():
    m = report["metrics"]
    print(name, json.dumps({k: m[k] for k in ["n", "tp", "fp", "fn", "problem_precision", "problem_recall", "literal_evidence", "critical_recall"]}, ensure_ascii=False))


rules-v0 {"n": 12, "tp": 12, "fp": 1, "fn": 0, "problem_precision": 0.9230769230769231, "problem_recall": 1.0, "literal_evidence": {"valid": 13, "total": 13, "ratio": 1.0}, "critical_recall": {"found": 1, "total": 1, "ratio": 1.0}}
rules-v1 {"n": 12, "tp": 12, "fp": 0, "fn": 0, "problem_precision": 1.0, "problem_recall": 1.0, "literal_evidence": {"valid": 12, "total": 12, "ratio": 1.0}, "critical_recall": {"found": 1, "total": 1, "ratio": 1.0}}
challenge {"n": 8, "tp": 0, "fp": 1, "fn": 5, "problem_precision": 0.0, "problem_recall": 0.0, "literal_evidence": {"valid": 1, "total": 1, "ratio": 1.0}, "critical_recall": {"found": 0, "total": 1, "ratio": 0.0}}


In [3]:
for name, report in reports.items():
    print("\n", name, "— ошибки:")
    for error in report["metrics"]["errors"]:
        print(error["id"], json.dumps(error["differences"], ensure_ascii=False))



 rules-v0 — ошибки:
gd-0005 {"sentiment": {"expected": "mixed", "actual": "negative"}}
gd-0006 {"problems": {"missing": [], "extra": ["material_quality"]}, "severity": {"expected": "none", "actual": "high"}, "owner": {"expected": null, "actual": "seller"}, "sentiment": {"expected": "positive", "actual": "negative"}}

 rules-v1 — ошибки:
gd-0005 {"sentiment": {"expected": "mixed", "actual": "negative"}}

 challenge — ошибки:
ch-01 {"problems": {"missing": [], "extra": ["smell"]}, "severity": {"expected": "none", "actual": "medium"}, "owner": {"expected": null, "actual": "seller"}, "sentiment": {"expected": "positive", "actual": "negative"}}
ch-02 {"problems": {"missing": ["material_quality"], "extra": []}, "severity": {"expected": "high", "actual": "none"}, "owner": {"expected": "seller", "actual": null}, "sentiment": {"expected": "negative", "actual": "neutral"}}
ch-03 {"problems": {"missing": ["delivery_delay"], "extra": []}, "severity": {"expected": "medium", "actual": "none"}, "own

## Расчёт бюджета — предположение, не usage API
План: 10 порций × (1200 инструкция + 3000 данные), 10 × 2400 выход.
Курс 100 ₽/$ — условный параметр, не найденный текущий курс.
Тарифы и ссылки: `docs/model-candidates.md`.


In [4]:
incoming, outgoing = 10 * (1200 + 3000), 10 * 2400
yandex_rub = incoming / 1000 * 0.10 + outgoing / 1000 * 0.20
gemini_usd = incoming / 1_000_000 * 0.25 + outgoing / 1_000_000 * 1.50
print("Вход/выход:", incoming, outgoing)
print("Alice Flash, ₽:", round(yandex_rub, 2), "; полный повтор:", round(2*yandex_rub, 2))
print("Gemini, $:", round(gemini_usd, 4), "; при условных 100 ₽/$:", round(gemini_usd*100, 2))
print("Токены и стоимость облачного API: НЕ ИЗМЕРЕНЫ; локальные измерения ниже")


Вход/выход: 42000 24000
Alice Flash, ₽: 9.0 ; полный повтор: 18.0
Gemini, $: 0.0465 ; при условных 100 ₽/$: 4.65
Токены и стоимость облачного API: НЕ ИЗМЕРЕНЫ; локальные измерения ниже


## Вывод по правилам
100% на знакомом наборе не доказывают обобщение. На challenge правила пропустили пять проблем.
Ниже — результаты настоящих LLM-прогонов, сохранённые отдельными скриптами.


## Реальные локальные прогоны Qwen3-4B-4bit
Три промпта × 20 отзывов × 3 повтора = 180 генераций, плюс отдельный smoke.
Загрузка весов и генерация не выполняются повторно при запуске ноутбука.
Метрики считаются по сохранённым ответам; инструкция запуска — `experiments/lab1/README.md`.
P2 настроен после ошибок на этих данных: результаты не являются независимым тестом.


In [5]:
from run import evaluate, sha
from local_trials import file_digest
from summarize_local import summarize
base = root / "docs/experiments/results/lab1"
comparisons = []
for session_name in ["local-qwen3-4b-pilot", "local-qwen3-4b-p2"]:
    session = base / session_name
    assert json.loads((session / "completed.json").read_text())["status"] == "complete"
    for summary_path in session.glob("*/summary.json"):
        dataset = DATA if summary_path.parent.name.startswith("golden") else root / "experiments/lab1/challenge-v0.1.jsonl"
        stored = json.loads(summary_path.read_text())
        actual = evaluate(read_jsonl(dataset), read_jsonl(summary_path.parent / "responses.jsonl"))
        assert actual == stored["metrics"]
        assert sha(dataset) == stored["metadata"]["dataset_sha256"]
    comparisons.append(summarize(session))
print("Пересчёт всех метрик по реальным ответам: OK")
print("Основных генераций:", sum(c["total_calls"] for c in comparisons))


Пересчёт всех метрик по реальным ответам: OK
Основных генераций: 180


In [6]:
for comparison in comparisons:
    for g in comparison["groups"]:
        m = g["first_run_metrics"]
        print(g["dataset"], g["prompt"], "n=", g["unique_records"],
              "precision=", round(m["problem_precision"], 3),
              "recall=", round(m["problem_recall"], 3),
              "schema=", round(m["schema_valid"], 3),
              "quotes=", m["literal_evidence"], "critical=", m["critical_recall"],
              "mean_seconds=", round(g["mean_latency_seconds"], 2),
              "MLX_peak_GB=", round(g["max_mlx_memory_gb"], 2),
              "stable=", g["predictions_identical_across_repeats"])
print("Стоимость облачного API: 0 ₽. Электричество и амортизация не измерены.")


challenge p0 n= 8 precision= 0.429 recall= 0.6 schema= 1.0 quotes= {'valid': 7, 'total': 7, 'ratio': 1.0} critical= {'found': 0, 'total': 1, 'ratio': 0.0} mean_seconds= 2.18 MLX_peak_GB= 3.06 stable= True
challenge p1 n= 8 precision= 0.667 recall= 0.8 schema= 0.75 quotes= {'valid': 5, 'total': 6, 'ratio': 0.8333333333333334} critical= {'found': 1, 'total': 1, 'ratio': 1.0} mean_seconds= 3.06 MLX_peak_GB= 3.06 stable= True
golden p0 n= 12 precision= 0.769 recall= 0.833 schema= 1.0 quotes= {'valid': 12, 'total': 13, 'ratio': 0.9230769230769231} critical= {'found': 0, 'total': 1, 'ratio': 0.0} mean_seconds= 2.67 MLX_peak_GB= 3.06 stable= True
golden p1 n= 12 precision= 0.75 recall= 0.75 schema= 0.833 quotes= {'valid': 8, 'total': 12, 'ratio': 0.6666666666666666} critical= {'found': 0, 'total': 1, 'ratio': 0.0} mean_seconds= 3.33 MLX_peak_GB= 3.06 stable= True
challenge p2 n= 8 precision= 0.667 recall= 0.8 schema= 1.0 quotes= {'valid': 6, 'total': 6, 'ratio': 1.0} critical= {'found': 0, 't

## Решение и ограничения
P2 исправил JSON null и улучшил часть метрик, но critical пропущен, есть ошибки цитат.
Выбран подход: структурированное извлечение + внешние проверки + агрегация кодом.
Qwen3-4B-4bit — доступный локальный baseline; production-модель ещё не выбрана.
Группировка внутри товара, полный пакет 100 отзывов, North Star и независимый safety-eval не проверены.
Три greedy-повтора с одинаковыми метками показывают воспроизводимость, не независимую статистику.
